# 02 - Baselines and reference forecasts

Persistence, smart persistence and the classical ML family, on exactly the
split every other experiment uses.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)


In [ ]:
registry = pd.read_csv(ROOT / "results" / "experiments.csv")
names = ["persistence", "smart_persistence", "linear_regression",
         "random_forest", "gradient_boosting", "xgboost"]
baselines = registry[registry.model.isin(names)]
display(baselines[["model", "horizon", "mae", "rmse", "nrmse_capacity",
                   "r2", "skill_vs_persistence_rmse",
                   "train_seconds"]].drop_duplicates("model"))


Smart persistence is the last observation rescaled by the clear-sky
ratio, with a reference floor, a ratio clip and a capacity clip. That
ratio is what makes the reference strong: a naive persistence is beaten
by almost anything, so a skill score against it flatters every model.


In [ ]:
from solar_forecasting.models.baselines import smart_persistence_forecast
last = np.array([40_000.0])
print(smart_persistence_forecast(last, np.array([700.0]), np.array([900.0]),
                                   rated_w=55_000.0))
print(smart_persistence_forecast(last, np.array([0.0]), np.array([0.0]),
                                   rated_w=55_000.0),
      "<- below the reference floor, so plain persistence")


In [ ]:
preds = pd.read_parquet(ROOT / "results" / "predictions" /
                            "B_ml_vs_dl__persistence__1h.parquet")
daylight = preds[preds["daylight"].astype(bool)]
print(len(daylight), "daylight steps")
display(daylight[["actual", "predicted", "error"]].describe())


## What to take from this notebook

- The reference is the point of comparison, not a formality. Skill is
  measured against both persistence variants, on identical timestamps.
- A negative skill score is a reportable result, not a failure of the run.
